# 分类决策：固定留出分数、阈值与误差分析

本 Notebook 使用 Phase 29 冻结的 prediction handoff。所有阈值比较只发生在 validation；选定阈值锁定后，test 只汇总一次。

In [1]:
from pathlib import Path
import json

root = Path.cwd()
while not (root / 'public' / 'classification' / 'phase-30').exists() and root != root.parent:
    root = root.parent
package = root / 'public' / 'classification' / 'phase-30'
sweep = json.loads((package / 'outputs' / 'threshold-sweep.json').read_text())
roc = json.loads((package / 'outputs' / 'roc.json').read_text())
decision = json.loads((package / 'outputs' / 'cost-selection.json').read_text())
errors = json.loads((package / 'outputs' / 'subgroup-errors.json').read_text())

summary = {
    'validation_rows': sum(decision['validation']['confusion'].values()),
    'selected_threshold': decision['selectedThreshold'],
    'validation_confusion': decision['validation']['confusion'],
    'validation_precision': decision['validation']['metrics']['precision'],
    'validation_recall': decision['validation']['metrics']['recall'],
    'validation_f1': decision['validation']['metrics']['f1'],
    'validation_auc': roc['auc'],
    'fold_threshold_range': decision['variation'],
    'locked_test_confusion': decision['lockedTest']['confusion'],
    'test_evaluations': decision['testEvaluations'],
    'named_validation_errors': len(errors['namedErrors']),
}
summary


{'validation_rows': 206,
 'selected_threshold': 0.09,
 'validation_confusion': {'fn': 1, 'fp': 2, 'tn': 113, 'tp': 90},
 'validation_precision': 0.9782608695652174,
 'validation_recall': 0.989010989010989,
 'validation_f1': 0.9836065573770493,
 'validation_auc': 0.9994266602962255,
 'fold_threshold_range': {'maximum': 0.5, 'minimum': 0.01},
 'locked_test_confusion': {'fn': 1, 'fp': 4, 'tn': 110, 'tp': 91},
 'test_evaluations': 1,
 'named_validation_errors': 3}

## 已发布结果

ROC/AUC 描述跨阈值排序，不是阈值选择器。特征子组仅用于教学诊断，不是人口属性公平性审计。